## Lab 04

## Zadanie 1 (Igor Sieradzki)

Funkcja <t>flow_rate</i> do modyfikacji

In [1]:
def flow_rate(weight, time, *, period=1, units_per_kg=1):
    """Funkcja wylicza ile wagi produktu przybyło/ubyło w jednostce czasu."""
    return weight / time * period * units_per_kg

weight = 0.5
time = 3

In [2]:
flow = flow_rate(weight, time)
print("{0:.3} kg per second".format(flow))

0.167 kg per second


In [3]:
"""Zmodyfikuj funkcję flow_rate tak, aby poniższe wywołania działały poprawnie."""

'Zmodyfikuj funkcję flow_rate tak, aby poniższe wywołania działały poprawnie.'

In [4]:
flow = flow_rate(weight, time, period=60, units_per_kg=1000)
print("{} grams per minute".format(flow))

10000.0 grams per minute


In [5]:
flow = flow_rate(weight, time, period=1, units_per_kg=1)
print("{0:.3} kg per second".format(flow))

0.167 kg per second


In [6]:
flow = flow_rate(weight, time)
print("{0:.3} grams per minute".format(flow))

0.167 grams per minute


Wyjaśnić czemu poniższe rzucanie błędu jest porządane

In [7]:
try:
    flow = flow_rate(weight, time, 3600, 2.2)
except TypeError:
    print(True)

True


## Zadanie 2 (Igor Sieradzki)

Dopisz definicje dekoratora _timeit_, który wypisze na wyjscie standardowe czas wywołania udekorowanej funckji np.

```
@timeit
def foo(x):
    return x**2
    
r = foo(2)
```
wypize: <br>
    `Function foo took: 0.00001 seconds` <br>
oraz zwróci 4

In [8]:
from functools import wraps
from time import time

def timeit(func):
    """Wypisuje czas wywołania udekorowanej funckji."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time()
        result = func(*args, **kwargs)
        print(f"Function {func.__name__} took: {time() - start:.5f} seconds")
        return result
    return wrapper

@timeit
def squares_list(n):
    squares = []
    for i in range(n):
        squares.append(i ** 2)
    return squares

@timeit
def squares_comprehension(n):
    return [i ** 2 for i in range(n)]

@timeit
def squares_map(n):
    return map(lambda x: x**2, range(n))

n = 1000000
l = squares_list(n)
c = squares_comprehension(n)
m = squares_map(n)

Function squares_list took: 0.02904 seconds
Function squares_comprehension took: 0.02455 seconds
Function squares_map took: 0.00000 seconds


## Zadanie 3 (Igor Sieradzki)

Dopisz definicje dekoratora _derivate_ wg. instrukcji w _docstringu_

In [9]:
import sys
sys.float_info.epsilon # epsilon maszynowy


def derivate(epsilon=None):
    """
    Zwraca pochodną funkcji w punkcie, wg. wzoru f'(x) = [f(x+h) - f(x)]/h,
    gdzie h jest parametrem dekoratora, jeśli nie zostanie podany, należy przyjąć 1000 * epsilon maszynowy
    """
    step = 1000 * sys.float_info.epsilon if epsilon is None else epsilon

    def decorator(function):
        @wraps(function)
        def derivative(x):
            return (function(x + step) - function(x)) / step
        return derivative
    return decorator


@derivate(0.01)
def f(x):
    return x*x

@derivate(0.00001)
def g(x):
    return x*x*x+3

def test(a, b, eps=1):
    return abs(round(a)-round(b)) < eps

print(test(f(100), 200.0))
print(round(f(0)) == 0.0)

print(test(g(100), 30000.0))
print(round(g(0)) == 0.0)

from random import random
for x in [random()*1000. for _ in range(20)]:
    print(f(x), 2*x, '\t', test(f(x), 2*x))
    print(g(x), 3*x**2, '\t', test(g(x), 3*x**2))

True
True
True
True
1458.5154139669612 1458.505413971771 	 True
1595428.5502433775 1595428.5319387256 	 True
1720.3717447118834 1720.3617447178058 	 True
2219733.428955078 2219733.3995163697 	 True
1778.3065676107071 1778.2965676124234 	 True
2371754.026412964 2371754.011786595 	 True
24.550641257010852 24.540641257009497 	 True
451.6826730650791 451.6823049789275 	 True
174.82134437732384 174.81134437726942 	 True
22919.25721801817 22919.254592241214 	 True
1739.4636533921584 1739.453653396001 	 True
2269274.2824554443 2269274.2592345215 	 True
246.43008532966633 246.42008532966474 	 True
45542.14754607528 45542.143840409444 	 True
1082.4773608648684 1082.467360867867 	 True
878801.7034530638 878801.6905081838 	 True
1913.638275465928 1913.6282754719427 	 True
2746479.904651642 2746479.8825142914 	 True
1896.2787235504948 1896.2687235555657 	 True
2696876.323223114 2696876.303951291 	 True
880.0408102339134 880.0308102368264 	 True
580840.6814932823 580840.6702245639 	 True
1051.15988

## Zadanie 4 (Igor Sieradzki)

Dopisz definicje dekoratora _accepts_ wg. instrukcji w _docstringu_

In [10]:
import inspect

def accepts(*types):
    """Sprawdza czy udekorowanej funckji zostały podane odpowiednie parametry zdefiniowane 
       w argumentach dekoratora"""
    def decorator(function):
        signature = inspect.signature(function)

        @wraps(function)
        def wrapper(*args, **kwargs):
            bound = signature.bind(*args, **kwargs)
            for parameter, expected_type in zip(signature.parameters, types):
                if parameter in bound.arguments and not isinstance(bound.arguments[parameter], expected_type):
                    raise TypeError(f"{parameter} must be {expected_type.__name__}")
            return function(*args, **kwargs)
        return wrapper
    return decorator


@accepts(str)
def capitalize(word):
    return word[0].upper() + word[1:]

print(capitalize('ola') == 'Ola')

try:
    capitalize(2)
except TypeError:
    print(True)

@accepts(float, int)
def static_pow(base, exp):
    return base ** exp 

print(static_pow(2., 2) == 4.)
print(static_pow(2., exp=2) == 4.)
print(static_pow(base=2., exp=2) == 4.)

try:
    static_pow('x', 10)
except TypeError:
    print(True)
    
try:
    static_pow(2, 2.2)
except TypeError:
    print(True)

True
True
True
True
True
True
True


## Zadanie 5 (Igor Sieradzki)

Dopisz definicje dekoratora _returns_ wg. instrukcji w _docstringu_

In [11]:
def returns(*types):
    """Sprawdza czy udekorowana funkcja zwraca poprawne argumenty, zdefiniowane w parametrach dekoratora"""
    def decorator(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            result = function(*args, **kwargs)
            results = result if len(types) > 1 else (result,)
            if len(results) != len(types) or any(not isinstance(value, expected_type) for value, expected_type in zip(results, types)):
                raise TypeError("Function returned an invalid type")
            return result
        return wrapper
    return decorator


@returns(str)
def str_only_identity(word):
    return word

print(str_only_identity('hello') == 'hello')

try:
    str_only_identity(10)
except TypeError:
    print(True)
    
@returns(int, int)
def split_indices(x):
    return x[0], x[1]

print(split_indices(x=[6,9]) == (6,9))

try:
    split_indices('AB')
except TypeError:
    print(True)

True
True
True
True


## Zadanie 6 (Igor Sieradzki)
Stwórz dekorator cached służący do cachowania wywołań dowolnej funkcji, tzn. chcemy by:
* wywołanie funkcji z określonymi argumentami miało miejsce tylko raz
* funkcja mogła przyjmować dowolną liczbę nazwanych i nienazwanych argumentów
* nie musi reagować poprawnie na domyślne argumenty, tzn. wywołanie funkcji z domyślnymi argumentami a podanie dokładnie takich samych może być traktowane jako dwa różne wywołania
* na opakowanej funkcji można wywołać `.cache_reset()`, który usunie cache z pamięci
* wywołanie `.cache_status()` zwraca string z opisem w postaci: <br>
    `Function FUNCTION_NAME called X times, evaluated Y times`

In [12]:
from random import random

def cached(function):
    cache = {}
    calls = 0
    evaluations = 0

    @wraps(function)
    def wrapper(*args, **kwargs):
        nonlocal calls, evaluations
        calls += 1
        key = args, tuple(sorted(kwargs.items()))
        if key not in cache:
            evaluations += 1
            cache[key] = function(*args, **kwargs)
        return cache[key]

    def cache_reset():
        cache.clear()

    def cache_status():
        return f"Function {function.__name__} called {calls} times, evaluated {evaluations} times"

    wrapper.cache_reset = cache_reset
    wrapper.cache_status = cache_status
    return wrapper

@cached
def foo(x, y=1, z=4):
    return random()
        
print(foo(3) == foo(3))
print(foo(4) == foo(4))
print(foo(3, z=-1, y=3) == foo(3, y=3, z=-1))
print(foo(3) != foo(x=3))
a = foo(3)
foo.cache_reset()
print(a != foo(3))
print(foo.cache_status() == 'Function foo called 10 times, evaluated 5 times')

True
True
True
True
True
True


## Zadanie 7 (Krzysztof Hajto)

Napisz dekorator który będzie robić n-krotne złożenie funkcji, gdzie n jest parametrem dekoratora

In [13]:
def zlozenie(n):
    def decorator(function):
        @wraps(function)
        def wrapper(value):
            for _ in range(n):
                value = function(value)
            return value
        return wrapper
    return decorator

@zlozenie(3)
def f1(x):
    return x+1

@zlozenie(2)
def f2(x):
    return x*x

@zlozenie(5)
def f3(word):
    return "".join(chr(ord(l)+1) for l in word)

print(f1(2)==5)
print(f2(3)==81)
print(f3("alamakota")=="fqfrfptyf")

True
True
True


## Zadanie 8 (Krzysztof Hajto)

Python nie ma wbudowanej instrukcji switch. Ale posiada anonimowe funkcje oraz słowniki. Zaimplementuj poniższy switch w postaci słownika funkcji.
`
int my_function(x, y) {


    switch(x) {
        case 1: return y*y;
        case 2: return x+y;
        case 3: return x*y;
        case 4: return 0;
    }
}
`

PS. Nigdy nie róbcie tego w faktycznym kodzie :)

In [14]:
def my_function(x, y):
    return {1: lambda: y * y, 2: lambda: x + y, 3: lambda: x * y, 4: lambda: 0}[x]()

print(my_function(1,3)==9)
print(my_function(2,4)==6)
print(my_function(3,1)==3)
print(my_function(4,9)==0)

True
True
True
True
